# Notebook Base — Desafío Tabular

Plantilla del equipo. El recorrido es el mismo siempre: leer la métrica,
auditar los datos, buscar trampas, modelar, decidir según el costo, enviar,
documentar.

Lo único que cambia entre un desafío y otro está en la celda **Configuración**.
Todo lo demás corre igual.

## Entorno

In [ ]:
import os, sys, subprocess
REPO = "https://github.com/AcostaAlex10/hackathon-kit-Acosta-Borges-Pelinski.git"
RAMA = "herramientas"

if not os.path.isdir("hackathon-kit-Acosta-Borges-Pelinski"):
    subprocess.run(["git", "clone", "-b", RAMA, REPO, "hackathon-kit-Acosta-Borges-Pelinski"], check=True)
os.chdir("hackathon-kit-Acosta-Borges-Pelinski") if os.path.basename(os.getcwd()) != "hackathon-kit-Acosta-Borges-Pelinski" else None
sys.path.insert(0, os.getcwd())

!pip -q install -r requirements.txt

from ic_kit.checkpoints import montar_drive
montar_drive()
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

## Datos

Ajustar según cómo los entregue la cátedra. Los notebooks base usan `gdown` o
`wget` contra Drive.

In [ ]:
!wget -c --no-check-certificate "PEGAR_URL_DE_LA_CATEDRA" -O data.zip
!unzip -o -qq data.zip

from pathlib import Path
DATA = Path("./data")
print([p.name for p in DATA.iterdir()])

## Configuración

La única celda que se toca al empezar. `CLASES` va **en el orden exacto de las
filas y columnas de la matriz de costos de la consigna**, no en orden
alfabético: si no coinciden, la penalización se aplica a la celda equivocada,
el código corre sin error y el puntaje baja sin explicación aparente.

In [ ]:
import numpy as np, pandas as pd
from ic_kit import costs, cleaning, tabular, traps, eda
from ic_kit import submit as sub_mod
from ic_kit.bitacora import Bitacora
from ic_kit.checkpoints import oof_con_checkpoint

RANDOM_STATE = 42
TARGET  = "nivel_urgencia"
ID_COL  = "id_paciente"

CLASES = ["No urgente", "Urgente", "Muy urgente", "Crítico"]

COSTO = np.array([
    [0,  1, 2, 4],
    [2,  0, 1, 3],
    [4,  2, 0, 2],
    [10, 6, 2, 0],
], dtype=float)

MENOR_ES_MEJOR = False       # True si la métrica es el costo directo

assert COSTO.shape == (len(CLASES), len(CLASES)), "la matriz no coincide con las clases"
assert (np.diag(COSTO) == 0).all(), "la diagonal debería ser cero"
bit = Bitacora()
pd.DataFrame(COSTO, index=CLASES, columns=CLASES).astype(int)

## Carga

In [ ]:
train = pd.read_csv(DATA / "train_labeled.csv")
test  = pd.read_csv(DATA / "test_features.csv")
try:
    unlab = pd.read_csv(DATA / "train_unlabeled.csv")
except FileNotFoundError:
    unlab = None

print("train", train.shape, "| test", test.shape,
      "| unlabeled", None if unlab is None else unlab.shape)

vistas = set(train[TARGET].astype(str))
faltan = vistas - set(map(str, CLASES))
assert not faltan, "etiquetas del train que no están en CLASES: %s" % faltan
print("las etiquetas del train coinciden con CLASES")
train.head()

## Exploración

`EDA.todo()` responde las preguntas que cambian decisiones bajo costo
asimétrico: cómo se reparte el target y cuánta exposición al costo aporta cada
clase, qué columnas traen anomalías, qué variables separan la clase cara, si
train y test se parecen, y qué costo dan las estrategias triviales. Guarda las
figuras numeradas para el informe.

In [ ]:
e = eda.EDA(train, target=TARGET, test=test, C=COSTO,
            clases=CLASES, id_col=ID_COL, figs="work/figs")
e.todo()

Anotar acá las hipótesis, antes de comprobarlas. Si después se
caen, esa contradicción vale para el informe.

In [ ]:
bit.hipotesis("...")

## Anomalías del conjunto de datos

Antes de entrenar. Un modelo entrenado sobre una fuga da la mejor validación
cruzada del aula y el peor puntaje del ranking.

In [ ]:
X0, y0, Xte0, _, _ = tabular.prepare(train, test, TARGET, ID_COL, class_order=CLASES)
diag = traps.run_all(X0, y0, Xte0, df_raw=train, id_col=ID_COL)

In [ ]:
# Registrar cada hallazgo con su evidencia; alimenta el probatorio.
bit.hallazgo("...", evidencia="...", decision="...")

## Limpieza

No se imputa: los árboles manejan faltantes de forma nativa y el patrón de qué
falta suele portar señal. Se agregan indicadores de ausencia.

In [ ]:
DESCARTAR = []          # columnas con fuga, constantes o vacías en test
RANGOS    = {}          # {"temperatura": (30, 45)} -> fuera de rango a NaN

trc, _ = cleaning.auto_clean(train, target=TARGET, drop_cols=DESCARTAR, ranges=RANGOS)
tec, _ = cleaning.auto_clean(test,  drop_cols=DESCARTAR, ranges=RANGOS, verbose=False)

# Mezcla de unidades, si la auditoría la marcó:
# for d in (trc, tec):
#     d["temperatura"] = cleaning.fix_units(d["temperatura"], 5/9, -32*5/9, threshold=50)

X, y, Xte, ids, clases = tabular.prepare(trc, tec, TARGET, ID_COL, class_order=CLASES)
print("features:", X.shape[1])

## Modelo

`oof_con_checkpoint` guarda después de cada fold. Si Colab se desconecta,
reejecutar esta celda retoma donde quedó en vez de empezar de nuevo.

In [ ]:
oof, pte, info = oof_con_checkpoint(
    X, y, Xte, C=COSTO, n_splits=5, seeds=(42, 43, 44), nombre="oof_principal")
info

## Decisión sensible al costo

Con una matriz de costos asimétrica, la clase más probable no es la decisión
que minimiza la pérdida esperada. La regla óptima es

$$\hat{y}(x) = \arg\min_j \sum_i P(y=i \mid x)\, C_{ij}$$

La exactitud baja y el costo mejora: se sacrifican aciertos en confusiones
baratas para evitar las caras.

In [ ]:
print(costs.decision_gain(y, oof, COSTO))
print()
print(costs.confusion_cost_report(y, costs.bayes_decision(oof, COSTO), COSTO, labels=clases))

Dos ajustes que a veces suman y a veces no. Ambos se comprueban
antes de aplicarse, porque los dos pueden empeorar el resultado.

In [ ]:
traps.fit_prior_honest(oof, y, COSTO)
traps.validate_prior_em(oof / oof.sum(1, keepdims=True), y)

USAR_PESOS = False       # poner True sólo si fit_prior_honest dio ganancia real
USAR_EM    = False       # poner True sólo si validate_prior_em dijo que sirve

p_final = pte.copy()
if USAR_PESOS:
    w = tabular.fit_prior(oof, y, COSTO)
    p_final = p_final * w
if USAR_EM:
    prior_tr = np.bincount(y, minlength=len(clases)) / len(y)
    _, p_final = traps.estimate_test_prior(p_final / p_final.sum(1, keepdims=True), prior_tr)

pred = costs.bayes_decision(p_final, COSTO)

## Envío

In [ ]:
s = sub_mod.make_submission(ids, pred, clases, "work/submit.csv",
                            id_col=ID_COL, target_col=TARGET)
dist_train = pd.Series(np.bincount(y, minlength=len(clases)) / len(y), index=clases)
sub_mod.validate(s, expected_ids=ids, allowed_labels=clases, train_dist=dist_train)

In [ ]:
log = sub_mod.SubmitLog(lower_is_better=MENOR_ES_MEJOR)
print(log.can_submit())

# Después de subirlo al servidor:
# log.record("work/submit.csv", cv=info["costo_oof"], notes="baseline mínimo costo")
# log.set_lb(1, PUNTAJE_DEL_RANKING)
# log.report()

Antes de gastar un intento, comparar contra el envío anterior.
Si cambia menos del 1 % de las filas, no aporta información nueva.

In [ ]:
# sub_mod.diff_vs("work/submit.csv", "work/submit_anterior.csv")

## Probatorio

In [ ]:
np.save("work/oof.npy", oof); np.save("work/y.npy", y)

from ic_kit.probatorio import generar_notebook
generar_notebook(
    grupo="...", integrantes=["Acosta", "Borges", "Pelinski"],
    desafio="...", metrica="...",
    oof="work/oof.npy", y="work/y.npy", C=COSTO, clases=clases)

In [ ]:
e.zip()          # figuras del EDA
bit.mostrar()    # revisar qué secciones quedaron vacías